# 01 Ingest Raw CSV To Parquet

Цель этапа: потоково прочитать большой `events.csv`, привести базовые типы и сохранить данные в Parquet shards.

Вход:

- `data/extracted/events.csv`

Выход:

- `data/interim/events_shard_*.parquet`
- `artifacts/manifests/ingest_manifest.json`

По умолчанию notebook стартует в `DRY_RUN` на одном chunk, чтобы быстро проверить схему. Для полного прогона переключите `DRY_RUN = False` в контрольной ячейке.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.io_utils import ingest_csv_to_parquet, load_ingest_config

config = load_ingest_config(PROJECT_ROOT / "configs" / "data.yaml")
config

IngestConfig(raw_csv_path='data/extracted/events.csv', interim_dir='data/interim', manifest_path='artifacts/manifests/ingest_manifest.json', encoding='utf-8-sig', chunksize=1000000, compression='zstd', row_group_size=100000, user_col='appmetrica_device_id', event_name_col='event_name', event_json_col='event_json', event_datetime_col='event_datetime', event_timestamp_col='event_timestamp', session_id_col='session_id')

In [3]:
raw_csv_path = PROJECT_ROOT / config.raw_csv_path
raw_csv_size_gb = raw_csv_path.stat().st_size / 1024**3

print(f"CSV path: {raw_csv_path}")
print(f"CSV size: {raw_csv_size_gb:.2f} GB")
print(f"Chunk size: {config.chunksize:,}")

CSV path: /root/projects/bert4rec/data/extracted/events.csv
CSV size: 18.22 GB
Chunk size: 1,000,000


## Control Cell

Сначала выполните `DRY_RUN = True`: он создаст один shard и manifest. Если всё выглядит нормально, поменяйте на `False` и перезапустите ingestion cell ниже для полного прогона.

In [6]:
DRY_RUN = False
MAX_CHUNKS = 1 if DRY_RUN else None

print("DRY_RUN:", DRY_RUN)
print("MAX_CHUNKS:", MAX_CHUNKS)

DRY_RUN: False
MAX_CHUNKS: None


In [7]:
manifest = ingest_csv_to_parquet(
    config=config,
    project_root=PROJECT_ROOT,
    max_chunks=MAX_CHUNKS,
)

print("num_shards:", manifest["num_shards"])
print("rows_in:", manifest["rows_in"])
print("rows_out:", manifest["rows_out"])
print("rows_dropped:", manifest["rows_dropped"])
manifest["shards"][:3]

CSV chunks: 0it [00:00, ?it/s]

num_shards: 74
rows_in: 73063802
rows_out: 73063802
rows_dropped: 0


[{'shard_idx': 0,
  'path': 'data/interim/events_shard_00000.parquet',
  'rows_in': 1000000,
  'rows_out': 1000000,
  'rows_dropped': 0},
 {'shard_idx': 1,
  'path': 'data/interim/events_shard_00001.parquet',
  'rows_in': 1000000,
  'rows_out': 1000000,
  'rows_dropped': 0},
 {'shard_idx': 2,
  'path': 'data/interim/events_shard_00002.parquet',
  'rows_in': 1000000,
  'rows_out': 1000000,
  'rows_dropped': 0}]

In [ ]:
import pandas as pd

first_shard = PROJECT_ROOT / manifest["shards"][0]["path"]
sample = pd.read_parquet(first_shard).head(5)

print(first_shard)
sample

In [ ]:
manifest_path = PROJECT_ROOT / config.manifest_path
print(f"Manifest written to: {manifest_path}")
print(f"First shard: {first_shard}")